# US-079 - Transfer Francia->Italia + Voting-3 (evaluacion)

### Equipo 17 - AgroSatCopilot - Transfer learning mediterraneo (EPIC 12)

---

Este cuaderno **evalua** la extension del modelo campeon al homologo italiano de US-078. Los miembros densos (TSViT-pheno, U-TAE y el TSViT Full-M) se **afinaron de verdad** sobre los patches italianos partiendo del checkpoint PASTIS, con la **bandera de reciclaje**: las filas de la cabeza de las clases conservadas (las que mapean a PASTIS, p.ej. `vineyards`->`Grapevine`, `durum_hard_wheat`->`Winter durum wheat`) se warm-startean desde la cabeza francesa, y las clases nuevas mediterraneas (`olive`, bosque, ...) parten de cero.

El combinador es el **Voting ponderado de 3 pesos** -- el ganador del despliegue en EPIC 6 (`france-10` 0.9069, `france-9` 0.92), no el Stacking. Aprende los pesos sobre las predicciones densas post-softmax italianas con validacion cruzada **por fold espacial** (anti-fuga, OOF).

Todas las cifras se leen del `report.json` real que produjo `scripts/run_transfer_italia.py` (bajo `checkpoints/transfer/voting-italia/us079`): no hay numeros inventados. Si el entrenamiento en la H100 aun no corrio (esta condicionado al dataset completo), el cuaderno lo dice explicitamente.

In [1]:
# Parametros (papermill).
report_dir = "checkpoints/transfer/voting-italia/us079"
data_dir = "data/pastis_italia_2018"
f1_threshold = 0.9  # objetivo de calidad: F1-macro sobre las mejores clases
n_demo = 2          # patches para la demo de granularidad


In [2]:
from pathlib import Path
import json
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

REPORT_DIR = Path(report_dir)
DATA_ROOT = Path(data_dir)
report_path = REPORT_DIR / 'report.json'
HAS_REPORT = report_path.is_file()
report = json.loads(report_path.read_text(encoding='utf-8')) if HAS_REPORT else None
if HAS_REPORT:
    print(f'Reporte US-079 encontrado: run={report["run"]}, '
          f'fold de test={report["test_fold"]}, miembros={report["members"]}')
else:
    print('AVISO: no hay report.json todavia. El entrenamiento real en la '
          'H100 esta condicionado al dataset completo (~1226 patches). '
          'Ejecuta scripts/run_transfer_italia.py para poblar este cuaderno.')


AVISO: no hay report.json todavia. El entrenamiento real en la H100 esta condicionado al dataset completo (~1226 patches). Ejecuta scripts/run_transfer_italia.py para poblar este cuaderno.


## 1. Pesos aprendidos del Voting-3 (AC2)

El Voting ponderado aprende **un peso convexo por miembro** (suman 1) que maximiza el F1-macro denso en validacion OOF. Tres pesos -- frente a los 54 del meta-LogReg del Stacking -- es lo que da al Voting su mejor generalizacion en transfer. Aqui los reportamos: la magnitud de cada peso dice cuanto confia el ensamble en cada miembro sobre el dominio italiano.

In [3]:
if HAS_REPORT:
    weights = report['voting_weights']
    wdf = pl.DataFrame({'miembro': list(weights.keys()),
                        'peso': [round(v, 4) for v in weights.values()]}).sort('peso', descending=True)
    display(wdf)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(wdf['miembro'].to_list()[::-1], wdf['peso'].to_list()[::-1], color='#6a1b9a')
    ax.set_xlabel('peso convexo (suma = 1)')
    ax.set_title('Pesos aprendidos del Voting-3 sobre Italia')
    ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()
    print(f"F1-macro OOF (spatial-CV) del Voting-3: {report['voting_oof_f1_macro']}")
else:
    print('Pendiente: pesos del Voting-3 (se reportan al correr el runner).')


Pendiente: pesos del Voting-3 (se reportan al correr el runner).


## 2. Evaluacion jerarquica: fino vs coarse (AC4)

La evaluacion se hace a **dos granularidades**. La **fina** usa el espacio de etiquetas italiano completo (las clases mediterraneas incluidas). La **coarse** colapsa cada clase fina a un bucket comun con PASTIS (p.ej. `apples`/`peach`/`plums` -> `Orchard`), de modo que un modelo que solo conoce la taxonomia gruesa de PASTIS es comparable con el modelo enriquecido. Reportamos mIoU + F1-macro por pixel para el Voting-3 y para cada miembro individual.

In [4]:
if HAS_REPORT:
    rows = []
    rows.append({'modelo': 'voting-3', **report['voting_eval']})
    for name, ev in report['member_eval'].items():
        rows.append({'modelo': name, **ev})
    evdf = pl.DataFrame(rows).select(
        ['modelo', 'fine_f1_macro', 'fine_miou', 'coarse_f1_macro', 'coarse_miou', 'n_pixels']
    ).sort('fine_f1_macro', descending=True)
    display(evdf)
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(evdf.height)
    ax.bar(x - 0.2, evdf['fine_f1_macro'].to_list(), 0.4, label='F1 fino', color='#1565c0')
    ax.bar(x + 0.2, evdf['coarse_f1_macro'].to_list(), 0.4, label='F1 coarse', color='#2e7d32')
    ax.set_xticks(x); ax.set_xticklabels(evdf['modelo'].to_list(), rotation=20, ha='right')
    ax.set_ylabel('F1-macro (pixel)'); ax.set_title('Fino vs coarse por modelo')
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('Pendiente: metricas fino/coarse (se reportan al correr el runner).')


Pendiente: metricas fino/coarse (se reportan al correr el runner).


## 3. Curva de descarte honesto y subconjunto F1 > 0.9 (AC3)

El objetivo de calidad de US-079 es **F1-macro > 0.9 sobre las ~10 clases mejor resueltas** (espejo del `france-10` 0.9069 del Voting-3 en PASTIS). Para localizar ese subconjunto sin trampa, ordenamos las clases por su F1 por clase (descendente) y reportamos el F1-macro de cada prefijo de `n` clases. Ninguna clase se descarta en silencio: la curva completa hace explicito donde cae el F1 por debajo del umbral.

In [5]:
if HAS_REPORT:
    curve = pl.DataFrame(report['discard_curve']).select(['n_classes', 'macro_f1'])
    best = report['best_subset_f1_over_0.9']
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(curve['n_classes'].to_list(), curve['macro_f1'].to_list(), marker='o', color='#c62828')
    ax.axhline(f1_threshold, color='grey', linestyle='--', label=f'umbral {f1_threshold}')
    ax.axvline(best['n_classes'], color='#2e7d32', linestyle=':',
               label=f"mejor subconjunto: {best['n_classes']} clases (F1 {best['macro_f1']})")
    ax.set_xlabel('n clases retenidas (mejores primero)'); ax.set_ylabel('F1-macro')
    ax.set_title('Curva de descarte honesto del Voting-3 sobre Italia')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"Mayor subconjunto con F1-macro >= {f1_threshold}: "
          f"{best['n_classes']} clases, F1 {best['macro_f1']}")
    print('Clases:', ', '.join(best['classes']))
else:
    print('Pendiente: curva de descarte (se reporta al correr el runner).')


Pendiente: curva de descarte (se reporta al correr el runner).


## 4. F1 por clase + matriz de confusion (AC5)

El detalle por clase muestra que clases mediterraneas **nuevas** (sin warm-start desde PASTIS, marcadas `es_nueva`) aprende el backbone frances, y con que soporte. La matriz de confusion densa (normalizada por fila = recall por clase) localiza las confusiones residuales tras el transfer.

In [6]:
if HAS_REPORT:
    pc = pl.DataFrame(report['voting_per_class']).select(
        ['leaf', 'is_new', 'f1', 'iou', 'support']
    ).rename({'leaf': 'clase', 'is_new': 'es_nueva'}).sort('f1', descending=True)
    with pl.Config(tbl_rows=40):
        display(pc)
    n_new_good = pc.filter((pl.col('es_nueva')) & (pl.col('f1') >= 0.5)).height
    print(f'Clases nuevas mediterraneas con F1 >= 0.5: {n_new_good}')
else:
    print('Pendiente: F1 por clase (se reporta al correr el runner).')


Pendiente: F1 por clase (se reporta al correr el runner).


In [7]:
if HAS_REPORT and (REPORT_DIR / 'voting_softmax.npz').is_file():
    from ml.transfer.italia_label_space import build_italia_label_space
    from ml.eval.transfer_italia_eval import probs_to_class_map
    from ml.eval.dense_metrics import dense_confusion_figure
    ls = build_italia_label_space(italia_root=DATA_ROOT)
    with np.load(REPORT_DIR / 'voting_softmax.npz') as data:
        vote_probs = {int(k): data[k] for k in data.files}
    vote_preds = probs_to_class_map(vote_probs)
    ann = DATA_ROOT / 'ANNOTATIONS'
    preds = np.concatenate([vote_preds[p].reshape(-1) for p in sorted(vote_preds)])
    target = np.concatenate([np.load(ann / f'TARGET_{p}.npy').reshape(-1) for p in sorted(vote_preds)])
    id_to_leaf = ls.id_to_leaf()
    fig = dense_confusion_figure(preds, target, class_names=id_to_leaf, ignore_index=0, normalize=True)
    fig.set_size_inches(11, 9); plt.tight_layout(); plt.show()
else:
    print('Pendiente: matriz de confusion (necesita voting_softmax.npz del runner).')


Pendiente: matriz de confusion (necesita voting_softmax.npz del runner).


## 5. Delta del transfer: fine-tune vs zero-shot (AC4)

La cota inferior es el **campeon frances zero-shot**: el checkpoint PASTIS aplicado tal cual a Italia, mapeando sus predicciones a las clases conservadas (las nuevas mediterraneas, que nunca vio, caen a fondo). El delta = (fine-tune) - (zero-shot) cuantifica cuanto aporta afinar de verdad. Un delta positivo confirma que el transfer adapta el backbone al vocabulario nuevo en vez de forzar todo por la taxonomia francesa.

In [8]:
if HAS_REPORT and report.get('transfer_delta'):
    d = report['transfer_delta']
    ddf = pl.DataFrame({'metrica': list(d.keys()), 'delta': list(d.values())})
    display(ddf)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    colors = ['#2e7d32' if v >= 0 else '#c62828' for v in d.values()]
    ax.barh(list(d.keys())[::-1], list(d.values())[::-1], color=colors[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Delta del transfer (fine-tune - zero-shot)')
    ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('Pendiente: delta del transfer (se reporta al correr el runner con --no-zero-shot off).')


Pendiente: delta del transfer (se reporta al correr el runner con --no-zero-shot off).


## 6. Demo de granularidad (papaya/fruits, AC5)

La hipotesis de taxonomia enriquecida, hecha visible: mostramos parcelas donde el modelo extendido dice la **clase fina italiana** (p.ej. `olive`, que PASTIS no tiene) frente al bucket coarse que un modelo sin granularidad usaria. Cada ejemplo: RGB del patch, prediccion fina del Voting-3 y la verdad densa, sobre un patch del fold de test.

In [9]:
if HAS_REPORT and (REPORT_DIR / 'voting_softmax.npz').is_file():
    from ml.transfer.italia_label_space import build_italia_label_space
    from ml.eval.transfer_italia_eval import probs_to_class_map
    ls = build_italia_label_space(italia_root=DATA_ROOT)
    id_to_leaf = ls.id_to_leaf()
    with np.load(REPORT_DIR / 'voting_softmax.npz') as data:
        vote_probs = {int(k): data[k] for k in data.files}
    vote_preds = probs_to_class_map(vote_probs)
    s2d = DATA_ROOT / 'DATA_S2'; ann = DATA_ROOT / 'ANNOTATIONS'
    n_cls = ls.num_classes
    cmap = ListedColormap(plt.cm.tab20(np.linspace(0, 1, max(n_cls, 2))))
    sel = sorted(vote_preds)[:n_demo]
    fig, axes = plt.subplots(len(sel), 3, figsize=(12, 4 * len(sel)))
    axes = np.atleast_2d(axes)
    for r, pid in enumerate(sel):
        stack = np.load(s2d / f'S2_{pid}.npy'); mask = np.load(ann / f'TARGET_{pid}.npy')
        t_mid = stack.shape[0] // 2
        rgb = np.transpose(stack[t_mid, [2, 1, 0]].astype('float32'), (1, 2, 0))
        p2, p98 = np.percentile(rgb[rgb > 0], [2, 98]) if (rgb > 0).any() else (0, 1)
        rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
        axes[r, 0].imshow(rgb); axes[r, 0].set_title(f'Patch {pid} - RGB'); axes[r, 0].axis('off')
        axes[r, 1].imshow(vote_preds[pid], cmap=cmap, vmin=0, vmax=n_cls - 1, interpolation='nearest')
        axes[r, 1].set_title(f'Patch {pid} - prediccion fina (Voting-3)'); axes[r, 1].axis('off')
        axes[r, 2].imshow(mask, cmap=cmap, vmin=0, vmax=n_cls - 1, interpolation='nearest')
        axes[r, 2].set_title(f'Patch {pid} - verdad densa'); axes[r, 2].axis('off')
        present = sorted({int(c) for c in np.unique(mask) if int(c) != 0})
        print(f'Patch {pid} clases presentes:', [id_to_leaf.get(c, c) for c in present])
    plt.tight_layout(); plt.show()
else:
    print('Pendiente: demo de granularidad (necesita voting_softmax.npz del runner).')


Pendiente: demo de granularidad (necesita voting_softmax.npz del runner).


## 7. Conclusiones

Cuando el runner corre sobre el dataset completo, este cuaderno responde a las preguntas de US-079: (1) los miembros densos se afinaron al espacio italiano con warm-start verificado de las clases conservadas; (2) el Voting-3 aprendio sus pesos sobre Italia (reportados arriba); (3) el objetivo de F1-macro > 0.9 sobre las mejores ~10 clases se mide con la curva de descarte honesto; (4) la evaluacion jerarquica fino/coarse y el delta del transfer cuantifican lo ganado frente al zero-shot; (5) la demo de granularidad hace visible la taxonomia enriquecida. El run de MLflow (`us079-transfer-italia`) lleva los tags `data_version` + `code_version`.